# Mushroom — Random Forest

Eén modeltype per notebook. Oorspronkelijke Mushroom-modellen: Andrew; de gedeelde
vergelijking en deze reproduceerbare versie: Codex op verzoek van Jorre.
Jorre rondt de definitieve taakverdeling en GenAI-policy later af (stap 5).

De huidige referentie is de correct uitgevoerde vergelijking op dezelfde 4.100
trainingsrijen, 900 ontwikkeltestrecords en vijf folds. Het model wordt hieronder
werkelijk opnieuw gefit met de vastgelegde configuratie. Alle preprocessing wordt
binnen de fold geleerd. De ontwikkeltest is eerder bekeken en is geen nieuwe
onafhankelijke eindtest. Dit notebook wijzigt de website of bronstudie niet.

We laden de vastgelegde parameters en leggen uit waarom dit model is opgenomen.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'MushroomFinal/mushroom_project_dataset.csv').exists())
sys.path.insert(0, str(ROOT))
summary = json.loads((ROOT/'MushroomFinal/comparison_correct/comparison_summary.json').read_text(encoding='utf-8'))
key = 'rf_final'
display(pd.Series(summary['classifier_parameters'][key], name='Vastgelegde parameters'))


bootstrap                       True
ccp_alpha                        0.0
class_weight                balanced
criterion                       gini
max_depth                         20
max_features                    None
max_leaf_nodes                  None
max_samples                     None
min_impurity_decrease            0.0
min_samples_leaf                   1
min_samples_split                  4
min_weight_fraction_leaf         0.0
monotonic_cst                   None
n_estimators                     300
n_jobs                             2
oob_score                      False
random_state                      42
verbose                            0
warm_start                     False
Name: Vastgelegde parameters, dtype: object

## Oorspronkelijk tuningonderzoek

Deze zoekruimte komt uit het oorspronkelijke projectnotebook. De oorspronkelijke
uitvoer blijft daar bewaard. De eerdere proximity-imputatie was vóór CV geleerd;
daarom gebruiken we die scores niet als eindbewijs. De gecorrigeerde vijf-fold-
evaluatie hieronder leert preprocessing opnieuw binnen iedere fold. Dit is een
reproductie van de vastgelegde configuratie, geen nieuwe gridsearch of modelkeuze.

```python
param_grid = {
    "classifier__n_estimators": [200, 300],
    "classifier__max_depth": [20, 30, None],
    "classifier__min_samples_split": [2, 3, 4],
    "classifier__min_samples_leaf": [1, 2],
    "classifier__max_features": ["sqrt", None]
}
```

Bomen kunnen niet-lineaire interacties modelleren. RF300 was de keuze op gedeelde CV macro F1; giftige recall en concrete fouten worden apart gerapporteerd.

We fitten vijf trainingsfolds en daarna één model op alle trainingsrijen. De code controleert alle 900 labels en kansen tegen de opgeslagen referentie.

In [2]:
from Final.tools.model_workflow import reproduce
result = reproduce('rf_final')
display(pd.DataFrame(result['folds']))
display(pd.Series({k:v for k,v in result.items() if k != 'folds'}))
assert result['labels_match_original'] and result['max_probability_difference'] < 1e-12


rf_final fold 1 /5


rf_final fold 2 /5


rf_final fold 3 /5


rf_final fold 4 /5


rf_final fold 5 /5


,fold,accuracy,balanced_accuracy,macro_f1,precision_p,recall_p,f1_p,roc_auc,average_precision,tn,fp,fn,tp
0,1,0.779268,0.744651,0.753527,0.766393,0.601286,0.673874,0.805545,0.738313,452,57,124,187
1,2,0.790244,0.752241,0.763004,0.800866,0.594855,0.682657,0.825880,0.769191,463,46,126,185
2,3,0.779268,0.742853,0.752203,0.769874,0.593548,0.670310,0.812720,0.747045,455,55,126,184
3,4,0.820732,0.785674,0.798014,0.846809,0.641935,0.730275,0.841373,0.812797,474,36,111,199
4,5,0.769512,0.734377,0.742633,0.746939,0.590323,0.659459,0.795497,0.735577,448,62,127,183


model                          rf_final
cv_macro_f1_mean               0.761876
cv_macro_f1_std                0.021452
cv_recall_p_mean                0.60439
accuracy                       0.813333
balanced_accuracy              0.780541
macro_f1                       0.791372
precision_p                     0.82397
recall_p                       0.645161
f1_p                           0.723684
roc_auc                         0.86227
average_precision              0.815399
tn                                  512
fp                                   47
fn                                  121
tp                                  220
fit_and_cv_seconds            30.472721
training_rows                      4100
test_rows                           900
labels_match_original              True
max_probability_difference          0.0
sklearn                           1.4.2
dtype: object

De uiteindelijke keuze staat in `09_vergelijking.ipynb`. Een hoge score voor de eetbare klasse is geen bevestiging van eetbaarheid; de website gebruikt een afzonderlijke voorzichtige beslisregel.